# Full certificate-to-repair demonstration on the Llama 3.1 8B head

This notebook runs one locked setting from the pilot: **dispersed BF16 tamper, `N=256`, 64 changed weights, 16 data-selected erasures, and 114 protected checks**. Each check is a residue modulo $2^{31}-1$, so the serialized certificate contains exactly $114\times31=3{,}534$ meaningful bits and occupies **442 bytes** after byte padding.

The separation is deliberate:

- the **encoder**, before tampering, scans every BF16 weight label and writes only `certificate.bin`;
- the **decoder** receives the damaged BF16 head, 256 retained inputs and 8-bit output vectors, the public numerical settings, and those 442 bytes;
- the **evaluator** alone retains the simulated attack and the original-head hash.

Both the original and damaged syndromes are computed by full-head scans. No sparse-residual shortcut is used. The retained-output cache and the frozen rule come from `llama_repair_certificate_pilot.ipynb`; this notebook does not select any setting on this demonstration instance.

In [4]:
# Run once if needed.
# %pip install -q "torch>=2.4" "transformers>=4.48" pandas tqdm

In [5]:
from dataclasses import dataclass, asdict
from pathlib import Path
import gc, hashlib, json, math, time

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

from repair_decoder import P, DecodeFailure, decode_patch, syndrome as syndrome_cpu

@dataclass(frozen=True)
class Config:
    model_id: str = "meta-llama/Meta-Llama-3.1-8B-Instruct"
    hf_cache_dir: str = "/scratch/bbjr/skarmakar/huggingface"
    model_revision: str | None = None  # pin for archival runs
    cache_dir: str = "repair_runs/certificate_pilot"
    frozen_dir: str = "repair_runs/matched_precision"
    out_dir: str = "repair_runs/full_certificate_demo"
    N: int = 256
    sparsity: int = 64
    erasures: int = 16
    checks: int = 114
    output_bits: int = 8
    accumulation_bits: int = 32
    fixed_logit_range: float = 64.0
    rho_multiple: float = 10.0
    tamper_fraction_of_bound: float = 0.9
    constraint_slack: float = 1e-4
    solver_steps: int = 350
    solver_check_every: int = 50
    solver_tol: float = 1e-4
    head_batch: int = 32
    syndrome_chunk: int = 1_048_576
    benchmark_entries: int = 8_388_608
    max_estimated_minutes_per_scan: float = 30.0
    pilot_seed: int = 17
    retained_sample_seed: int = 100_017  # first locked test trial

CFG=Config()
assert torch.cuda.is_available(), "A CUDA GPU is required."
assert P==2**31-1 and CFG.checks*31==3534 and math.ceil(CFG.checks*31/8)==442
assert CFG.N==256 and CFG.sparsity==64 and CFG.erasures==16 and CFG.output_bits==8
DEVICE=torch.device("cuda")
torch.set_grad_enabled(False)
torch.backends.cuda.matmul.allow_tf32=False
torch.set_float32_matmul_precision("highest")
CACHE=Path(CFG.cache_dir); FROZEN=Path(CFG.frozen_dir); OUT=Path(CFG.out_dir)
OUT.mkdir(parents=True,exist_ok=True)
CERT_PATH=OUT/"certificate.bin"
print(torch.cuda.get_device_name())
print(asdict(CFG))

NVIDIA H200
{'model_id': 'meta-llama/Meta-Llama-3.1-8B-Instruct', 'hf_cache_dir': '/scratch/bbjr/skarmakar/huggingface', 'model_revision': None, 'cache_dir': 'repair_runs/certificate_pilot', 'frozen_dir': 'repair_runs/matched_precision', 'out_dir': 'repair_runs/full_certificate_demo', 'N': 256, 'sparsity': 64, 'erasures': 16, 'checks': 114, 'output_bits': 8, 'accumulation_bits': 32, 'fixed_logit_range': 64.0, 'rho_multiple': 10.0, 'tamper_fraction_of_bound': 0.9, 'constraint_slack': 0.0001, 'solver_steps': 350, 'solver_check_every': 50, 'solver_tol': 0.0001, 'head_batch': 32, 'syndrome_chunk': 1048576, 'benchmark_entries': 8388608, 'max_estimated_minutes_per_scan': 30.0, 'pilot_seed': 17, 'retained_sample_seed': 100017}


## 1. Load the locked artifacts and original BF16 head

The cached hidden states are exact FP32 representations of BF16 activations. The retained prediction for each input is the complete vocabulary logit vector quantized into public 8-bit cells over `[-64,64)`. Thus ordinary retained-output storage is `256 × vocab_size × 8` bits; it is reported separately from the protected certificate.

In [6]:
required=[CACHE/"hidden_cal.pt",CACHE/"hidden_test.pt",
          CACHE/"codes_test_w16_native_acc32_y8_v4.pt"]
missing=[str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Run the corrected matched-precision pilot first; missing: "+str(missing))

frozen_path=FROZEN/"frozen_choices.csv"
if frozen_path.exists():
    frozen=pd.read_csv(frozen_path)
    row=frozen[(frozen.weight_bits==16)&(frozen.layout=="dispersed")&(frozen.N==CFG.N)]
    if len(row)!=1:
        raise RuntimeError("Expected one frozen dispersed BF16 N=256 row.")
    locked=row.iloc[0]
    assert int(locked.certificate_checks)==CFG.checks, locked.to_dict()
    assert int(locked.chosen_erasures)==CFG.erasures, locked.to_dict()
else:
    print("Frozen CSV not found; using the published locked rule in Config.")

Hcal=torch.load(CACHE/"hidden_cal.pt",map_location="cpu",weights_only=True)
Htest=torch.load(CACHE/"hidden_test.pt",map_location="cpu",weights_only=True)
Ctest=torch.load(CACHE/"codes_test_w16_native_acc32_y8_v4.pt",map_location="cpu",weights_only=True)
assert Htest.ndim==2 and Hcal.shape[1]==Htest.shape[1] and len(Htest)>=CFG.N
assert Ctest.dtype==torch.uint8 and Ctest.shape[0]==len(Htest)
assert torch.equal(Hcal,Hcal.to(torch.bfloat16).float())
assert torch.equal(Htest,Htest.to(torch.bfloat16).float())

model=AutoModelForCausalLM.from_pretrained(
    CFG.model_id,cache_dir=CFG.hf_cache_dir,revision=CFG.model_revision,torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,attn_implementation="sdpa"
).eval().to(DEVICE)
assert hasattr(model,"lm_head") and model.lm_head.weight.dtype==torch.bfloat16
assert not bool(getattr(model.config,"tie_word_embeddings",True))
W=model.lm_head.weight.detach()
V,D=W.shape
assert D==Htest.shape[1] and Ctest.shape[1]==V and V*D<P
del model; gc.collect(); torch.cuda.empty_cache()

R=float(CFG.fixed_logit_range); WIDTH=2*R/(2**CFG.output_bits)
def qcode(z):
    if bool(((z < -R)|(z >= R)).any()):
        raise RuntimeError("Public logit range saturated; cache and settings disagree.")
    return torch.floor((z+R)/WIDTH).to(torch.uint8)

# Build public calibration pools and the INT8 reference changes once, then free FP32.
W32=W.float()
# The pilot created output caches in batches of 32. Reproduce that GEMM shape:
# changing the batch shape can change the FP32 reduction path by a few ulps and
# move logits lying exactly near an 8-bit cell boundary into an adjacent cell.
check_n=min(CFG.head_batch,len(Htest))
check=qcode(Htest[:check_n].to(DEVICE)@W32.T).cpu()
cache_mismatch=(check!=Ctest[:check_n])
if bool(cache_mismatch.any()):
    raise RuntimeError(
        f"Cached predictions still disagree when reproduced with batch size {check_n}: "
        f"{int(cache_mismatch.sum())} of {cache_mismatch.numel()} codes differ. "
        "The cache is stale or the unpinned model revision changed; do not continue. "
        "Recompute the pilot caches with a pinned model_revision."
    )
weight_rms=float(W32.square().mean().sqrt())
rho=CFG.rho_multiple*weight_rms
tamper_step=CFG.tamper_fraction_of_bound*rho
with torch.inference_mode():
    logits=Hcal.to(DEVICE)@W32.T
    row_pool=torch.topk(torch.softmax(logits,dim=-1).mean(0),min(256,V)).indices.cpu().numpy()
    col_pool=torch.topk(Hcal.square().mean(0),min(512,D)).indices.cpu().numpy()
    row_scale=(W32.abs().amax(1)/127).to(torch.bfloat16).float()
assert bool((row_scale>0).all())
del W32,logits,check; gc.collect(); torch.cuda.empty_cache()
print({"vocab":V,"hidden":D,"head_entries":V*D,"rho":rho,
       "ordinary_output_MiB":CFG.N*V*CFG.output_bits/8/2**20,
       "certificate_bytes":442})

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

{'vocab': 128256, 'hidden': 4096, 'head_entries': 525336576, 'rho': 0.1426707860082388, 'ordinary_output_MiB': 31.3125, 'certificate_bytes': 442}


## 2. Exact certificate utilities and preflight checks

A BF16 weight is labeled by its unsigned 16-bit storage pattern. Check `k` is $\sum_j x_j(j+1)^k\bmod(2^{31}-1)$. The GPU implementation uses only exact signed-64-bit operations. The preflight compares it with the independent CPU implementation and checks certificate packing byte for byte. It then benchmarks a prefix and refuses to start a full scan if the estimate exceeds the declared guardrail.

In [7]:
def bf16_labels_gpu(head):
    labels=head.view(torch.int16).reshape(-1).to(torch.int64)
    return labels.bitwise_and_(0xffff)

def _mod_product(a,b):
    v=a*b  # operands are residues; product is below 2**62
    v=(v & P)+(v >> 31)
    v=(v & P)+(v >> 31)
    return torch.where(v>=P,v-P,v)

def full_syndrome_gpu(labels,checks,chunk,progress=True):
    x=labels.reshape(-1)
    if x.dtype!=torch.int64 or not 0<=checks or len(x)>=P or not 0<chunk<=2**20:
        raise ValueError("Expected int64 labels, dimension < P, and chunk <= 2**20.")
    out=torch.zeros(checks,device=x.device,dtype=torch.int64)
    starts=range(0,len(x),chunk)
    if progress: starts=tqdm(starts,total=math.ceil(len(x)/chunk),desc="full-head syndrome")
    for start in starts:
        vals=x[start:start+chunk]
        a=torch.arange(start+1,start+1+len(vals),device=x.device,dtype=torch.int64)
        power=torch.ones_like(a)
        for k in range(checks):
            out[k]=(out[k]+_mod_product(vals,power).sum())%P
            if k+1<checks:
                power=_mod_product(power,a)
    return out

def pack_31(values):
    acc=0; nbits=0; out=bytearray()
    for value in np.asarray(values,dtype=np.int64):
        value=int(value)
        if not 0<=value<2**31: raise ValueError("A check is not a 31-bit value.")
        acc |= value << nbits; nbits += 31
        while nbits>=8:
            out.append(acc & 255); acc >>= 8; nbits -= 8
    if nbits: out.append(acc & 255)
    return bytes(out)

def unpack_31(blob,count):
    acc=0; nbits=0; values=[]
    for byte in blob:
        acc |= int(byte) << nbits; nbits += 8
        while nbits>=31 and len(values)<count:
            values.append(acc & (2**31-1)); acc >>= 31; nbits -= 31
    if len(values)!=count or acc!=0:
        raise ValueError("Malformed certificate or nonzero padding.")
    return np.asarray(values,dtype=np.int64)

def sha256_bf16(head,rows_per_chunk=2048):
    digest=hashlib.sha256(); raw=head.view(torch.int16)
    for start in range(0,len(raw),rows_per_chunk):
        digest.update(raw[start:start+rows_per_chunk].cpu().numpy().tobytes(order="C"))
    return digest.hexdigest()

# Independent exactness tests.
toy=bf16_labels_gpu(W[:1,:1024].contiguous())
gpu=full_syndrome_gpu(toy,9,1024,progress=False).cpu().numpy()
cpu=syndrome_cpu(toy.cpu().numpy(),9,chunk=1024)
assert np.array_equal(gpu,cpu)
test_values=np.random.default_rng(0).integers(0,P,size=CFG.checks,dtype=np.int64)
test_blob=pack_31(test_values)
assert len(test_blob)==442 and np.array_equal(unpack_31(test_blob,CFG.checks),test_values)

# Warm up and estimate one full scan using the real head and all 114 checks.
prefix_n=min(CFG.benchmark_entries,V*D)
prefix=W.view(torch.int16).reshape(-1)[:prefix_n].to(torch.int64).bitwise_and_(0xffff)
torch.cuda.synchronize(); start=time.perf_counter()
_ = full_syndrome_gpu(prefix,CFG.checks,CFG.syndrome_chunk,progress=False)
torch.cuda.synchronize(); benchmark_seconds=time.perf_counter()-start
estimated_scan_minutes=benchmark_seconds*(V*D/prefix_n)/60
del prefix,_; torch.cuda.empty_cache()
print({"preflight":"passed","benchmark_seconds":benchmark_seconds,
       "estimated_minutes_per_full_scan":estimated_scan_minutes})
if estimated_scan_minutes>CFG.max_estimated_minutes_per_scan:
    raise RuntimeError("Estimated full scan exceeds the safety limit. Inspect GPU utilization before raising the limit.")

{'preflight': 'passed', 'benchmark_seconds': 0.13788362313061953, 'estimated_minutes_per_full_scan': 0.14391603164258412}


## 3. Pre-tamper encoder: scan and serialize the original head

This is the only stage allowed to inspect the original head. The in-memory syndrome is deleted after packing, and the decoder later reloads only the 442-byte file. The SHA-256 hash is evaluator-only and is never passed to the decoder.

In [8]:
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
encode_base=torch.cuda.memory_allocated()
start=time.perf_counter()
original_labels=bf16_labels_gpu(W)
certificate_values=full_syndrome_gpu(original_labels,CFG.checks,CFG.syndrome_chunk)
torch.cuda.synchronize()
certificate_scan_seconds=time.perf_counter()-start
encode_peak_total=torch.cuda.max_memory_allocated()/2**30
encode_peak_incremental=(torch.cuda.max_memory_allocated()-encode_base)/2**30
certificate_values=certificate_values.cpu().numpy()
del original_labels; torch.cuda.empty_cache()

start=time.perf_counter()
certificate_blob=pack_31(certificate_values)
CERT_PATH.write_bytes(certificate_blob)
serialize_seconds=time.perf_counter()-start
assert len(certificate_blob)==442 and CERT_PATH.stat().st_size==442
assert np.array_equal(unpack_31(CERT_PATH.read_bytes(),CFG.checks),certificate_values)
del certificate_values,certificate_blob

start=time.perf_counter()
_evaluator_original_hash=sha256_bf16(W)
evaluator_hash_seconds=time.perf_counter()-start
public_metadata={
    "field_prime":P,"label_bits":16,"checks":CFG.checks,
    "meaningful_certificate_bits":CFG.checks*31,
    "serialized_certificate_bytes":len(CERT_PATH.read_bytes()),
    "dimension":V*D,"N":CFG.N,"output_bits":CFG.output_bits,
    "accumulation_bits":CFG.accumulation_bits,"erasures":CFG.erasures,
    "rho":rho,"solver_steps":CFG.solver_steps
}
(OUT/"public_settings.json").write_text(json.dumps(public_metadata,indent=2))
print({"certificate_scan_seconds":certificate_scan_seconds,
       "serialize_seconds":serialize_seconds,
       "certificate_bits":CFG.checks*31,"certificate_bytes":CERT_PATH.stat().st_size,
       "encoder_peak_allocated_GiB":encode_peak_total,
       "encoder_peak_incremental_GiB":encode_peak_incremental})

full-head syndrome:   0%|          | 0/501 [00:00<?, ?it/s]

{'certificate_scan_seconds': 8.655663697980344, 'serialize_seconds': 0.004402864025905728, 'certificate_bits': 3534, 'certificate_bytes': 442, 'encoder_peak_allocated_GiB': 4.971189498901367, 'encoder_peak_incremental_GiB': 3.9609384536743164}


## 4. Apply one predetermined dispersed BF16 tamper

The evaluator uses the first locked test trial from the pilot—not a trial selected for favorable damage or recovery. It first generates legal INT8 numerical changes, then adds those same signed changes to the BF16 head and rounds once to BF16. The attack record below remains evaluator-only. The decoder is called through an interface that does not accept it.

In [9]:
_EVALUATOR_TAMPER_TRIAL_ID=1000  # predetermined first fresh test trial

def make_evaluator_tamper():
    rng=np.random.default_rng(CFG.pilot_seed+10_000*_EVALUATOR_TAMPER_TRIAL_ID)
    rows=rng.choice(row_pool,CFG.sparsity,replace=False).astype(np.int64)
    cols=rng.choice(col_pool,CFG.sparsity,replace=True).astype(np.int64)
    signs=rng.choice([-1,1],CFG.sparsity).astype(np.int64)
    rr=torch.as_tensor(rows,device=DEVICE); cc=torch.as_tensor(cols,device=DEVICE)
    scale=row_scale[rr]
    old_signed=torch.round(W[rr,cc].float()/scale).clamp(-127,127).to(torch.int64)
    step=torch.round(torch.full_like(scale,tamper_step)/scale).clamp_min(1).to(torch.int64)
    step=torch.minimum(step,torch.floor(torch.full_like(scale,rho)/scale).to(torch.int64))
    sign=torch.as_tensor(signs,device=DEVICE)
    proposed=(old_signed+sign*step).clamp(-127,127)
    unchanged=proposed==old_signed
    proposed=torch.where(unchanged,(old_signed-sign*step).clamp(-127,127),proposed)
    if bool((proposed==old_signed).any()): raise RuntimeError("INT8 reference change vanished.")
    reference_delta=(proposed-old_signed).float()*scale
    old=W[rr,cc].clone(); new=(old.float()+reference_delta).to(torch.bfloat16)
    actual_delta=new.float()-old.float()
    if bool((new==old).any()) or float(actual_delta.abs().max())>rho:
        raise RuntimeError("BF16 tamper violates the frozen contract.")
    old_labels=(old.view(torch.int16).to(torch.int64)&0xffff).cpu().numpy()
    index=rows*D+cols
    assert len(np.unique(index))==CFG.sparsity
    return {"rows":rows,"cols":cols,"index":index,"old":old,"new":new,
            "old_labels":old_labels,"actual_delta":actual_delta.cpu().numpy(),
            "mean_match_error":float((actual_delta-reference_delta).abs().mean()),
            "max_match_error":float((actual_delta-reference_delta).abs().max())}

_evaluator_tamper=make_evaluator_tamper()
_rr=torch.as_tensor(_evaluator_tamper["rows"],device=DEVICE)
_cc=torch.as_tensor(_evaluator_tamper["cols"],device=DEVICE)
with torch.no_grad(): W[_rr,_cc]=_evaluator_tamper["new"]
assert sha256_bf16(W)!=_evaluator_original_hash, "Tamper did not change the checkpoint hash."

rng=np.random.default_rng(CFG.retained_sample_seed)
order=rng.choice(len(Htest),CFG.N,replace=True)
retained_H=Htest[order].clone()
retained_codes=Ctest[torch.as_tensor(order)].clone()
print({"changed_weights":CFG.sparsity,
       "mean_abs_change":float(np.mean(np.abs(_evaluator_tamper["actual_delta"]))),
       "mean_match_error":_evaluator_tamper["mean_match_error"],
       "max_match_error":_evaluator_tamper["max_match_error"]})

{'changed_weights': 64, 'mean_abs_change': 0.058751143515110016, 'mean_match_error': 7.65528529882431e-05, 'max_match_error': 0.0003662109375}


## 5. Decoder: examples + serialized certificate → repaired head

The data step compares all vocabulary outputs, solves the same fixed 350-step interval problem on mismatching rows, and marks the 16 largest proposal coordinates as erasures. The decoder then scans the damaged head to form its syndrome, subtracts it from the unpacked certificate, algebraically decodes the patch, and writes recovered BF16 bit patterns into the head.

In [10]:
def current_codes(head32,h):
    chunks=[]
    for start in range(0,len(h),CFG.head_batch):
        hb=h[start:start+CFG.head_batch].to(DEVICE,dtype=torch.float32)
        chunks.append(qcode(hb@head32.T).cpu())
    return torch.cat(chunks)

def interval_proposal(Hx,lower,upper):
    if len(lower)==0: return torch.empty((0,Hx.shape[1]),device=Hx.device),0.0,0
    gram=Hx@Hx.T if Hx.shape[0]<=Hx.shape[1] else Hx.T@Hx
    norm=torch.linalg.eigvalsh(gram)[-1].clamp_min(1e-30).sqrt()
    step=.99/norm
    E=torch.zeros((len(lower),Hx.shape[1]),device=Hx.device,dtype=torch.float32)
    extra=E.clone(); dual=torch.zeros_like(lower)
    previous_top=None; stable_checks=0; violation=float("inf")
    for iteration in range(1,CFG.solver_steps+1):
        u=dual+step*(extra@Hx.T)
        dual=u-step*torch.maximum(lower,torch.minimum(upper,u/step))
        z=E-step*(dual@Hx)
        new=(z.sign()*(z.abs()-step).clamp_min(0)).clamp(-rho,rho)
        extra=2*new-E; E=new
        if iteration%CFG.solver_check_every==0 or iteration==CFG.solver_steps:
            values=E@Hx.T
            violation=torch.maximum((lower-values).clamp_min(0).max(),
                                    (values-upper).clamp_min(0).max()).item()
            keep=min(CFG.sparsity,E.numel())
            top=torch.sort(torch.topk(E.abs().flatten(),keep).indices).values
            stable_checks=stable_checks+1 if previous_top is not None and torch.equal(top,previous_top) else 0
            previous_top=top
            if iteration>=300 and violation<=CFG.solver_tol and stable_checks>=3: break
    return E,violation,iteration

def localize_erasures(head32,h,stored_codes):
    observed=current_codes(head32,h)
    mismatch=(stored_codes!=observed).any(0)
    bad_rows=torch.nonzero(mismatch,as_tuple=False).flatten().cpu().numpy()
    if not len(bad_rows):
        return np.empty(0,np.int64),{"bad_rows":0,"changed_output_cells":0,
                                         "solver_violation":0.0,"solver_iterations":0}
    Hx=h.to(DEVICE,dtype=torch.float32)
    rows=torch.as_tensor(bad_rows,device=DEVICE)
    cur=head32[rows]@Hx.T
    q=stored_codes[:,torch.as_tensor(bad_rows)].T.to(DEVICE,dtype=torch.float32)
    lower=(-R+q*WIDTH)-cur-CFG.constraint_slack
    upper=(-R+(q+1)*WIDTH)-cur+CFG.constraint_slack
    proposal,violation,iterations=interval_proposal(Hx,lower,upper)
    keep=min(CFG.erasures,proposal.numel())
    where=torch.topk(proposal.abs().flatten(),keep).indices
    local_row=(where//D).cpu().numpy(); col=(where%D).cpu().numpy()
    erasures=bad_rows[local_row].astype(np.int64)*D+col.astype(np.int64)
    return erasures,{"bad_rows":len(bad_rows),
                     "changed_output_cells":int((stored_codes!=observed).sum()),
                     "solver_violation":violation,"solver_iterations":iterations}

def apply_decoded_bf16_patch(head,indices,correction):
    idx=torch.as_tensor(indices,device=head.device,dtype=torch.int64)
    raw=head.view(torch.int16).reshape(-1)
    current=(raw[idx].to(torch.int64)&0xffff).cpu().numpy()
    recovered=(current+np.asarray(correction,dtype=np.int64))%P
    if np.any((recovered<0)|(recovered>=2**16)):
        raise DecodeFailure("Decoded values are not BF16 storage labels.")
    signed=np.where(recovered>=2**15,recovered-2**16,recovered).astype(np.int16)
    with torch.no_grad(): raw[idx]=torch.from_numpy(signed.copy()).to(head.device)

def repair_from_public_inputs(damaged_head,h,stored_codes,certificate_bytes):
    # Deliberately no original head, attack support, attack values, or attack seed argument.
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    base=torch.cuda.memory_allocated(); total_start=time.perf_counter()
    head32=damaged_head.float()  # working arithmetic; not protected information

    torch.cuda.synchronize(); start=time.perf_counter()
    erasures,diagnostics=localize_erasures(head32,h,stored_codes)
    torch.cuda.synchronize(); localization_seconds=time.perf_counter()-start

    torch.cuda.synchronize(); start=time.perf_counter()
    damaged_labels=bf16_labels_gpu(damaged_head)
    damaged_syndrome=full_syndrome_gpu(damaged_labels,CFG.checks,CFG.syndrome_chunk)
    torch.cuda.synchronize(); damaged_scan_seconds=time.perf_counter()-start
    damaged_syndrome=damaged_syndrome.cpu().numpy()
    del damaged_labels

    start=time.perf_counter()
    protected=unpack_31(certificate_bytes,CFG.checks)
    residual=(protected-damaged_syndrome)%P
    decoded_index,correction=decode_patch(
        residual,V*D,erasures=erasures,seed=CFG.pilot_seed
    )
    algebra_seconds=time.perf_counter()-start

    torch.cuda.synchronize(); start=time.perf_counter()
    apply_decoded_bf16_patch(damaged_head,decoded_index,correction)
    torch.cuda.synchronize(); patch_seconds=time.perf_counter()-start
    total_seconds=time.perf_counter()-total_start
    peak_total=torch.cuda.max_memory_allocated()/2**30
    peak_incremental=(torch.cuda.max_memory_allocated()-base)/2**30
    del head32; torch.cuda.empty_cache()
    return decoded_index,correction,erasures,{**diagnostics,
        "localization_seconds":localization_seconds,
        "damaged_syndrome_seconds":damaged_scan_seconds,
        "algebraic_decode_seconds":algebra_seconds,
        "patch_apply_seconds":patch_seconds,
        "total_repair_seconds":total_seconds,
        "decoder_peak_allocated_GiB":peak_total,
        "decoder_peak_incremental_GiB":peak_incremental}

certificate_bytes=CERT_PATH.read_bytes()
decoded_index,decoded_correction,selected_erasures,decoder_report=repair_from_public_inputs(
    W,retained_H,retained_codes,certificate_bytes
)
print(decoder_report)

full-head syndrome:   0%|          | 0/501 [00:00<?, ?it/s]

{'bad_rows': 64, 'changed_output_cells': 5934, 'solver_violation': 0.219346284866333, 'solver_iterations': 350, 'localization_seconds': 0.24558132491074502, 'damaged_syndrome_seconds': 8.843010221142322, 'algebraic_decode_seconds': 0.29263947415165603, 'patch_apply_seconds': 0.0006286220159381628, 'total_repair_seconds': 9.38513582595624, 'decoder_peak_allocated_GiB': 6.92822265625, 'decoder_peak_incremental_GiB': 5.917969703674316}


## 6. Evaluator-only verification and final report

The evaluator now reveals the simulated attack. Exact equality of the complete raw-head SHA-256 hashes is the primary restoration check. Support and old-label comparisons explain any failure; neither was available to the decoder.

In [11]:
start=time.perf_counter()
repaired_hash=sha256_bf16(W)
verification_seconds=time.perf_counter()-start
decoded_support_exact=(set(decoded_index.tolist())==set(_evaluator_tamper["index"].tolist()))
true_idx=torch.as_tensor(_evaluator_tamper["index"],device=DEVICE)
final_labels=(W.view(torch.int16).reshape(-1)[true_idx].to(torch.int64)&0xffff).cpu().numpy()
support_values_exact=np.array_equal(final_labels,_evaluator_tamper["old_labels"])
whole_head_exact=(repaired_hash==_evaluator_original_hash)
actual_success=bool(decoded_support_exact and support_values_exact and whole_head_exact)
caught=len(set(selected_erasures.tolist())&set(_evaluator_tamper["index"].tolist()))
errors_outside=CFG.sparsity-caught
required_checks=len(selected_erasures)+2*errors_outside

report={
    "model":CFG.model_id,"layout":"dispersed","weight_bits":16,
    "N":CFG.N,"changed_weights":CFG.sparsity,
    "output_bits":CFG.output_bits,"ordinary_output_MiB":CFG.N*V*CFG.output_bits/8/2**20,
    "certificate_checks":CFG.checks,"certificate_bits":CFG.checks*31,
    "certificate_bytes_with_padding":CERT_PATH.stat().st_size,
    "selected_erasures":len(selected_erasures),"true_support_caught":caught,
    "errors_outside_erasures":errors_outside,"required_checks_for_this_instance":required_checks,
    "decoded_patch_size":len(decoded_index),
    "mean_abs_weight_change":float(np.mean(np.abs(_evaluator_tamper["actual_delta"]))),
    "mean_match_error":_evaluator_tamper["mean_match_error"],
    "max_match_error":_evaluator_tamper["max_match_error"],
    "certificate_scan_seconds":certificate_scan_seconds,
    "certificate_serialize_seconds":serialize_seconds,
    "certificate_encoding_seconds":certificate_scan_seconds+serialize_seconds,
    "encoder_peak_allocated_GiB":encode_peak_total,
    "encoder_peak_incremental_GiB":encode_peak_incremental,
    **decoder_report,
    "verification_seconds":verification_seconds,
    "decoded_support_exact":decoded_support_exact,
    "support_values_exact":support_values_exact,
    "whole_head_SHA256_exact":whole_head_exact,
    "actual_decode_success":actual_success
}
if required_checks>CFG.checks:
    raise RuntimeError("This predetermined instance lies outside the frozen decoding radius.")
if not actual_success:
    raise RuntimeError("Repair failed evaluator verification: "+str(report))
pd.DataFrame([report]).to_csv(OUT/"full_demo_result.csv",index=False)
(OUT/"full_demo_result.json").write_text(json.dumps(report,indent=2))
display(pd.DataFrame([report]).T.rename(columns={0:"value"}))
print("PASS: decoder restored the complete BF16 head exactly.")

,value
model,meta-llama/Meta-Llama-3.1-8B-Instruct
layout,dispersed
weight_bits,16
N,256
changed_weights,64
output_bits,8
ordinary_output_MiB,31.3125
certificate_checks,114
certificate_bits,3534
certificate_bytes_with_padding,442


PASS: decoder restored the complete BF16 head exactly.


## What counts as success

A successful run must end with all three evaluator checks true: decoded support equals the simulated support, recovered BF16 labels equal their pre-tamper labels, and the SHA-256 hash of the entire repaired head equals the original hash. The runtime claim should use `certificate_encoding_seconds` and `total_repair_seconds`; the memory claim should use the two incremental GPU peaks. The 31.3125 MiB ordinary output record and the 442-byte protected certificate must remain separate in any table or paper text.